# Sorting on GPU

Sorting is the backbone of both intraday and EOD workflows, and is where the KDB-X GPU Edition really shines.
Two approaches are available depending on whether you need the full sorted result or just the top-N rows.
In this tutorial, we demonstrate sorting data both in-memory and on-disk.

## 1. Prerequisites

1. Requires KDB-X to be installed, you can sign up at https://developer.kx.com/products/kdb-x/install.
2. Ensure you are able to run `gentab.q`, which will generate dummy data for sorting.

## 2. Performing sorts on GPU

### Method 1: In-memory sort with `.gpu.iasc`

In-memory sorting can be achieved using `.gpu.iasc`

In [ ]:
.gpu.iasc[t]

where:
- `t` is a table on GPU

The return is an on GPU list of indexes given by sorting the table lexiocraphically based on column order.
To achieve a round trip from CPU, we can do:

In [ ]:
.gpu.from .gpu.iasc .gpu.to table

inMemSort:(`$string[l])!{T:.gpu.to t:?[`$"orders",string[x],"m";();0b;c!c:`orderId`time];
                tt1:.z.p;iasc t;tt1:.z.p-tt1;
                tt2:.z.p;.gpu.iasc T;tt2:.z.p-tt2;
                tt3:.z.p;.gpu.from .gpu.iasc .gpu.to t;tt3:.z.p-tt3;
                (tt1;tt2;tt3)} each l:10 20 30 40 50

### Method 2: Full table sort with `.gpu.xasc`

Below is an example of how to use this functionality on-disk - for wide tables, we are not currently seeing very good performance

In [ ]:
\l gentab.q

\d .sort

eq:{[d;c;j;i](=;c[i];d[i;j])}
start:{[j;d;c]({[d;c;j](eq[d;c;0] each til j),enlist[(<;c[j];d[j;0])]}[d;c]each til j+1),enlist (eq[d;c;0] each til j), enlist (&;(>=;c[j];d[j;0]);(<;c[j];d[j;1]))}
end:{[j;d;c]n:count flip d;enlist[(eq[d;c;n-2] each til j), enlist[(>=;c[j];d[j;n-1])]],({[d;c;j;n] (eq[d;c;n-2] each til j) ,enlist[(>;c[j];d[j;n-1])] }[d;c;;n] each reverse til j)};
stepipk:{[i;k;j;d;c] ({[d;c;j;I;i](eq[d;c;j]each til I+i+1),enlist(<;c[I+i+1];d[I+i+1;j])}[d;c;j;i] each til k-i),enlist (eq[d;c;j] each til k),enlist (&;(>=;c[k];d[k;j]);(<;c[k];d[k;j+1]))}
stepimk:{[i;k;j;d;c] enlist[({[d;c;j;i](=;c[i];d[i;j])}[d;c;j]each til i),enlist[(>=;c[i];d[i;j])]],({[d;c;j;I;i](eq[d;c;j]each til I-i+1),enlist ((>;c[I-i+1];d[I-i+1;j+1]))}[d;c;j;i] each til i-k+1), enlist (eq[d;c;j]each til k),enlist (&;(>;c[k];d[k;j]);(<;c[k];d[k;j+1]))   }
step:{$[0<=x[1]-x[0];stepipk[x[0];x[1]];stepimk[x[0];x[1]]]}
flipper:{$[type[x]in 11 20h;enlist each value x;x]}
statbatches:{ [t;c;M] labels:c,`xabcdefg;
		      L:asc (1000*M)?count[t];
		      a:labels xasc ?[t;enlist (in;`i;L);0b;labels!rc:c,`i];
		      d:value flip a floor count[a]*(1%M)*1+til M-1;
		      l:first each where each flip d <> next each d;
		      funcs:(start,step each -1_flip (-1_l;1_l)),end;
		      raze {z[0][z[1];x;y]}[flipper each d;rc] each flip (funcs;l[0],(1+til count[funcs]-2),last[-1_l])}

writebatch:{[x;o;y]
	upsert[o]each x
	}

sortsingle:{[c;t;x]
		.gpu.sdev[x[0]];
		fromdisk:?[d:?[t;x[1];0b;()];();0b;c!c];
		gputab:.gpu.to fromdisk;
		f:.gpu.iasc gputab;
		a:d@.gpu.from f;
		a}

.gpu.sort.xasc:{[c;t;o]

	// ensure data fits on GPUs 
	sizeData:sum hcount each `$string[t],/:string c;
	sizeGPU:min{.gpu.sdev[x];first .gpu.mdev[]}each til .gpu.ndev[];
	batchesGPU:ceiling 3*sizeData%sizeGPU;

	// ensure data fits on CPU
	sizeData:sum hcount each `$string[t],/:string cols get t;
	sizeCPU:(.Q.w[][`mphy])-.Q.w[][`heap];
	batchesCPU:ceiling 2*sizeData%sizeCPU%2*.gpu.ndev[];

	batches:statbatches[get t;c;max(3;batchesGPU;batchesCPU)];

	// function to upsert batches
	w:writebatch[;o;];

	// function to sort batches on multiple gpus
	s:{[x;t;c;y]
		sortsingle[c;t] peach flip(til count x;x)}[;t;c;];

	// main routine
	res:{[w;s;x;y]a:{x`}peach(w[x];s[y]);a 1}[w;s]/[();.gpu.ndev[]cut batches];
	upsert[o]each res;
	.Q.gc[];
  }

\d .

gentab[`:/app/example/db/orders10m/;10000000];
gentab[`:/app/example/db/orders20m/;20000000];
gentab[`:/app/example/db/orders30m/;30000000];
gentab[`:/app/example/db/orders40m/;40000000];
gentab[`:/app/example/db/orders50m/;50000000];

In-memory sort GPU is winning

For on-disk sort, times are bottlenecked by disk I/O